# 클러스터링 그리드 탐색 — 차원축소 x 모델 x 하이퍼파라미터

입력: 01_eda_preprocessing.ipynb가 저장한 중복 제거본 17,121건 (`cache/vectors_dedup.npy`, `cache/meta_dedup.csv`)

진행 방식 (차원축소와 클러스터링 파라미터는 상호작용하므로 분리하지 않고 한 판에 탐색한다):
1. UMAP 설정 x 모델 x 파라미터 전체 조합을 반복문으로 돌려 라벨 생성
2. 모든 조합을 하나의 잣대(원본 1024차원 코사인 실루엣)로 점수화
3. 상위 조합 5~10개로 좁힘
4. 후보들의 그룹 내용을 직접 읽고 최종 선택 (다음 노트북)

평가를 UMAP 공간이 아니라 원본 공간에서 하는 이유: UMAP은 이웃을 뭉치게 만드는 알고리즘이라, UMAP 공간에서 재면 "심하게 뭉치는 설정"이 무조건 이기는 순환 평가가 된다. 원본 공간에서 재면 모든 UMAP 설정이 같은 잣대로 비교된다.

## 0. 환경 확인

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import silhouette_score

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
print("환경 준비 완료")

환경 준비 완료


## 1. 데이터 로드

In [2]:
from pathlib import Path

CACHE = Path("cache")

# 01 노트북에서 저장한 중복 제거 확정본 (이미 L2 정규화된 벡터)
Xn = np.load(CACHE / "vectors_dedup.npy")
df = pd.read_csv(CACHE / "meta_dedup.csv")

print("벡터:", Xn.shape)
print("메타:", df.shape)
assert len(Xn) == len(df)

# 업종별 부분집합 - 벡터와 메타를 같은 마스크로 잘라 짝을 유지한다
subsets = {}
for cat in ["cnstwk", "frgcpt", "thng", "servc"]:
    m = (df["biz_div"] == cat).to_numpy()
    subsets[cat] = {"X": Xn[m], "df": df[m].reset_index(drop=True)}
    print(f"{cat}: {m.sum():,}건")

벡터: (17121, 1024)
메타: (17121, 4)
cnstwk: 1,526건
frgcpt: 313건
thng: 7,009건
servc: 7,513건


## 2. 그리드 탐색 정의

- UMAP: `n_components` 7가지 x `n_neighbors` 3가지 = 21회 (비싼 연산, 바깥 루프)
- `min_dist`는 0.0 고정 - 클러스터링 목적일 때의 표준. 점이 퍼지는 걸 막을 이유가 없다 (0.1~0.5는 시각화용)
- 모델: KMeans(K 3~15), HDBSCAN(min_cluster_size x min_samples) - 저차원에서 도는 안쪽 루프라 저렴
- Agglomerative는 제외 - 카테고리 계층 구조가 필요해지면 그때 추가한다
- 실루엣은 원본 1024차원 코사인 + 표본 2000건 (전체 쌍 계산은 실루엣이 UMAP보다 비싸지기 때문)

In [3]:
N_COMPONENTS_GRID = [5, 10, 15, 20, 30, 50, 100]
N_NEIGHBORS_GRID = [5, 15, 50]
KMEANS_K_GRID = list(range(3, 16))
# (min_cluster_size, min_samples)
HDBSCAN_GRID = [(mcs, ms) for mcs in (15, 30, 60) for ms in (5, 15)]


def evaluate_labels(X_orig, labels, sample_size=2000):
    """라벨 품질을 원본 공간 실루엣으로 잰다. 평가 불가능한 경우 None."""
    mask = labels >= 0  # HDBSCAN의 노이즈(-1)는 실루엣 계산에서 제외
    n_clusters = len(set(labels[mask]))
    if mask.sum() < 100 or n_clusters < 2:
        return None
    return silhouette_score(
        X_orig[mask], labels[mask], metric="cosine",
        sample_size=min(sample_size, int(mask.sum())), random_state=42,
    )


def run_grid(X_orig, category_name):
    """UMAP 21가지 x (KMeans 13 + HDBSCAN 6) 전체 조합을 돌리고 결과 표를 반환한다.

    임베딩은 반환하지 않는다 - random_state를 고정했으므로 상위 조합은
    다음 단계에서 같은 설정으로 재계산해 재현할 수 있다.
    """
    rows = []
    total = len(N_COMPONENTS_GRID) * len(N_NEIGHBORS_GRID)
    done = 0
    t0 = time.perf_counter()

    for n_comp in N_COMPONENTS_GRID:
        for n_nb in N_NEIGHBORS_GRID:
            emb = umap.UMAP(
                n_components=n_comp, n_neighbors=n_nb, min_dist=0.0,
                metric="cosine", random_state=42,
            ).fit_transform(X_orig)

            # KMeans - K를 안쪽 루프로 돌리면 UMAP 1회 비용으로 K 13가지를 전부 본다
            for k in KMEANS_K_GRID:
                labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(emb)
                sil = evaluate_labels(X_orig, labels)
                rows.append({
                    "category": category_name, "model": "kmeans",
                    "n_components": n_comp, "n_neighbors": n_nb,
                    "param": f"K={k}", "n_clusters": k,
                    "noise_ratio": 0.0, "silhouette": sil,
                })

            # HDBSCAN - K를 안 정하는 대신 밀도 파라미터를 돌린다. 노이즈 비율도 기록
            for mcs, ms in HDBSCAN_GRID:
                labels = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(emb)
                n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
                noise = float((labels == -1).mean())
                sil = evaluate_labels(X_orig, labels)
                rows.append({
                    "category": category_name, "model": "hdbscan",
                    "n_components": n_comp, "n_neighbors": n_nb,
                    "param": f"mcs={mcs},ms={ms}", "n_clusters": n_clusters,
                    "noise_ratio": round(noise, 3), "silhouette": sil,
                })

            done += 1
            print(f"  [{category_name}] UMAP {done}/{total} "
                  f"(n_comp={n_comp}, n_nb={n_nb}) 누적 {time.perf_counter()-t0:.0f}초", flush=True)

    return pd.DataFrame(rows)

## 3. cnstwk (공사, 1,526건)

가장 작고, 토목 지식으로 결과 검증이 가능한 업종이라 먼저 돌려서 그리드 구성 자체를 점검한다.

In [4]:
results_cnstwk = run_grid(subsets["cnstwk"]["X"], "cnstwk")
results_cnstwk.to_csv(CACHE / "grid_cnstwk.csv", index=False, encoding="utf-8-sig")
print(f"조합 {len(results_cnstwk)}개 완료")

c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PRO

  [cnstwk] UMAP 1/21 (n_comp=5, n_nb=5) 누적 40초


c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PRO

  [cnstwk] UMAP 2/21 (n_comp=5, n_nb=15) 누적 52초


c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PRO

...

  [cnstwk] UMAP 19/21 (n_comp=100, n_nb=5) 누적 295초


c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PRO

  [cnstwk] UMAP 20/21 (n_comp=100, n_nb=15) 누적 311초


c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PROJECTS\bidding-agent\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\user\Desktop\PRO

  [cnstwk] UMAP 21/21 (n_comp=100, n_nb=50) 누적 331초
조합 399개 완료


### 3-1. cnstwk 상위 조합

In [5]:
# HDBSCAN은 노이즈 비율 30% 초과 시 실루엣이 높아도 실용성이 없으므로 걸러낸다
valid = results_cnstwk[
    results_cnstwk["silhouette"].notna() & (results_cnstwk["noise_ratio"] <= 0.3)
]
valid.sort_values("silhouette", ascending=False).head(15)

,category,model,n_components,n_neighbors,param,n_clusters,noise_ratio,silhouette
32,cnstwk,hdbscan,5,15,"mcs=15,ms=5",38,0.272,0.114692
260,cnstwk,hdbscan,30,15,"mcs=15,ms=5",32,0.263,0.109507
146,cnstwk,hdbscan,15,15,"mcs=15,ms=5",33,0.256,0.108480
281,cnstwk,hdbscan,30,50,"mcs=30,ms=5",16,0.290,0.105362
317,cnstwk,hdbscan,50,15,"mcs=15,ms=5",33,0.202,0.104817
203,cnstwk,hdbscan,20,15,"mcs=15,ms=5",31,0.250,0.104299
165,cnstwk,hdbscan,15,50,"mcs=15,ms=5",27,0.285,0.101545
53,cnstwk,hdbscan,5,50,"mcs=30,ms=5",16,0.298,0.100456
167,cnstwk,hdbscan,15,50,"mcs=30,ms=5",16,0.270,0.099982
279,cnstwk,hdbscan,30,50,"mcs=15,ms=5",25,0.279,0.099242


## 4. frgcpt (외자, 313건)

In [ ]:
results_frgcpt = run_grid(subsets["frgcpt"]["X"], "frgcpt")
results_frgcpt.to_csv(CACHE / "grid_frgcpt.csv", index=False, encoding="utf-8-sig")
valid = results_frgcpt[
    results_frgcpt["silhouette"].notna() & (results_frgcpt["noise_ratio"] <= 0.3)
]
valid.sort_values("silhouette", ascending=False).head(15)

## 5. thng (물품, 7,009건)

In [ ]:
results_thng = run_grid(subsets["thng"]["X"], "thng")
results_thng.to_csv(CACHE / "grid_thng.csv", index=False, encoding="utf-8-sig")
valid = results_thng[
    results_thng["silhouette"].notna() & (results_thng["noise_ratio"] <= 0.3)
]
valid.sort_values("silhouette", ascending=False).head(15)

## 6. servc (용역, 7,513건)

In [ ]:
results_servc = run_grid(subsets["servc"]["X"], "servc")
results_servc.to_csv(CACHE / "grid_servc.csv", index=False, encoding="utf-8-sig")
valid = results_servc[
    results_servc["silhouette"].notna() & (results_servc["noise_ratio"] <= 0.3)
]
valid.sort_values("silhouette", ascending=False).head(15)

## 7. 결과 종합

In [ ]:
all_results = pd.concat(
    [results_cnstwk, results_frgcpt, results_thng, results_servc], ignore_index=True
)
all_valid = all_results[
    all_results["silhouette"].notna() & (all_results["noise_ratio"] <= 0.3)
]

# 업종별 최고 조합
best = all_valid.loc[all_valid.groupby("category")["silhouette"].idxmax()]
print("업종별 최고 조합:")
print(best.to_string(index=False))

In [ ]:
# 파라미터 민감도 확인 - n_components / n_neighbors 별로 최고 실루엣이 어떻게 변하는가
# 곡선이 평평하면 그 파라미터는 결과에 둔감했다는 뜻 (그리드에 넣은 게 과잉이었는지 사후 검증)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for cat in ["cnstwk", "frgcpt", "thng", "servc"]:
    sub = all_valid[all_valid["category"] == cat]
    by_comp = sub.groupby("n_components")["silhouette"].max()
    axes[0].plot(by_comp.index, by_comp.values, marker="o", label=cat)
    by_nb = sub.groupby("n_neighbors")["silhouette"].max()
    axes[1].plot(by_nb.index, by_nb.values, marker="o", label=cat)
axes[0].set_xlabel("n_components")
axes[0].set_ylabel("최고 실루엣")
axes[0].set_title("n_components 민감도")
axes[0].legend()
axes[1].set_xlabel("n_neighbors")
axes[1].set_title("n_neighbors 민감도")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# KMeans K별 최고 실루엣 - 엘보우 대신 쓰는 K 비교 곡선 (실루엣은 K끼리 직접 비교 가능)
plt.figure(figsize=(8, 4))
km = all_valid[all_valid["model"] == "kmeans"]
for cat in ["cnstwk", "frgcpt", "thng", "servc"]:
    sub = km[km["category"] == cat]
    by_k = sub.groupby("n_clusters")["silhouette"].max()
    plt.plot(by_k.index, by_k.values, marker="o", label=cat)
plt.xlabel("K")
plt.ylabel("최고 실루엣 (원본 공간)")
plt.title("KMeans K별 최고 실루엣")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. 결론

실행 후 채운다.

- 업종별 최고 조합 (UMAP 설정 / 모델 / 파라미터 / 실루엣):
- n_components 민감도 (그리드에 넣은 게 값을 했는가):
- n_neighbors 민감도:
- KMeans vs HDBSCAN 어느 쪽이 우세했는가 / 노이즈 비율은 어땠는가:
- 다음 단계(육안 검증)로 넘길 후보 조합: